# Cloudflare Realtime multi-camera RF-DETR Medium checkpoint inference

This Colab notebook is paired with the **Cloudflare Realtime Multi-Camera** HTML page.

The flow is:

1. The HTML page publishes two or more browser cameras into one Cloudflare Realtime source session.
2. Copy either the **JSON** or **ENV** source coordinates from the HTML page into this notebook.
3. The notebook subscribes to every listed camera track.
4. One shared **Roboflow RF-DETR Medium** model loads your trained `.pth` checkpoint and processes the newest frame from each camera.
5. All annotated outputs are published into one processed Cloudflare session, with one output track per camera.
6. Paste the processed `sessionId` and each processed `trackName` into the HTML result viewers, or consume the printed JSON from a Next.js backend.

RF-DETR performs object detection with bounding boxes. This notebook uses the current `rfdetr` checkpoint API rather than Ultralytics RT-DETR.

Documentation:

- https://rfdetr.roboflow.com/latest/reference/rfdetr/
- https://rfdetr.roboflow.com/latest/reference/medium/


## 1. Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU** before running the notebook.


In [ ]:
%pip install -q -U "rfdetr>=1.7.1,<2" supervision av aiortc aiohttp

In [ ]:
import asyncio
import getpass
import json
import re
import threading
import time
import uuid
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import aiohttp
import av
import numpy as np
import torch

from PIL import Image, ImageDraw, ImageFont
from rfdetr import RFDETR, RFDETRMedium

from aiortc import (
    MediaStreamTrack,
    RTCConfiguration,
    RTCIceServer,
    RTCPeerConnection,
    RTCSessionDescription,
    VideoStreamTrack,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. In Colab choose Runtime → Change runtime type → GPU."
    )

DEVICE = 0
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__)


## 2. Configure Cloudflare

Use the same Cloudflare Realtime application as the HTML page. Enter the **new/rotated** App Secret; it is hidden while typing.


In [ ]:
APP_ID = input(
    "Cloudflare Realtime App ID [08974e44564c2b4da09a97378a557e7f]: "
).strip() or "08974e44564c2b4da09a97378a557e7f"

APP_SECRET = getpass.getpass("NEW Cloudflare Realtime App Secret: ").strip()
if not APP_SECRET:
    raise ValueError("App Secret cannot be empty.")

print("Cloudflare App ID:", APP_ID)


## 3. Paste the source coordinates from the HTML page

On the HTML page:

1. Start and publish all cameras.
2. Click **Copy JSON format** or **Copy ENV format**.
3. Replace the placeholder inside `SOURCE_COORDINATES` below with the copied block.

Both formats produced by the HTML page are supported.


In [ ]:
SOURCE_COORDINATES = r"""
PASTE_THE_JSON_OR_ENV_BLOCK_FROM_THE_HTML_PAGE_HERE
""".strip()


def parse_source_coordinates(raw: str) -> dict[str, Any]:
    raw = raw.strip()
    if not raw or raw == "PASTE_THE_JSON_OR_ENV_BLOCK_FROM_THE_HTML_PAGE_HERE":
        raise ValueError(
            "Paste the JSON or ENV source coordinates copied from the HTML page."
        )

    if raw.startswith("{"):
        try:
            data = json.loads(raw)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSON source coordinates: {exc}") from exc

        session_id = str(data.get("sessionId", "")).strip()
        cameras_raw = data.get("cameras", [])
        if not isinstance(cameras_raw, list):
            raise ValueError("JSON field 'cameras' must be a list.")

        cameras = []
        for index, item in enumerate(cameras_raw, start=1):
            if not isinstance(item, dict):
                raise ValueError(f"cameras[{index - 1}] must be an object.")
            cameras.append(
                {
                    "name": str(item.get("name") or f"Camera {index}").strip(),
                    "trackName": str(item.get("trackName", "")).strip(),
                    "width": item.get("width"),
                    "height": item.get("height"),
                    "fps": item.get("fps"),
                }
            )
    else:
        env: dict[str, str] = {}
        for raw_line in raw.splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                raise ValueError(f"Invalid ENV line: {raw_line!r}")
            key, value = line.split("=", 1)
            env[key.strip()] = value.strip()

        session_id = env.get("SOURCE_SESSION_ID", "").strip()
        try:
            camera_count = int(env.get("SOURCE_CAMERA_COUNT", "0"))
        except ValueError as exc:
            raise ValueError("SOURCE_CAMERA_COUNT must be an integer.") from exc

        cameras = []
        for index in range(1, camera_count + 1):
            prefix = f"SOURCE_CAMERA_{index}_"
            cameras.append(
                {
                    "name": env.get(f"{prefix}NAME", f"Camera {index}").strip(),
                    "trackName": env.get(f"{prefix}TRACK_NAME", "").strip(),
                    "width": env.get(f"{prefix}WIDTH"),
                    "height": env.get(f"{prefix}HEIGHT"),
                    "fps": env.get(f"{prefix}FPS"),
                }
            )

    if not session_id:
        raise ValueError("The source sessionId is missing.")
    if not cameras:
        raise ValueError("No camera entries were found.")

    seen_track_names: set[str] = set()
    for index, camera in enumerate(cameras, start=1):
        if not camera["name"]:
            camera["name"] = f"Camera {index}"
        if not camera["trackName"]:
            raise ValueError(f"Camera {index} is missing trackName.")
        if camera["trackName"] in seen_track_names:
            raise ValueError(f"Duplicate source trackName: {camera['trackName']}")
        seen_track_names.add(camera["trackName"])

    return {"sessionId": session_id, "cameras": cameras}


SOURCE_CONFIG = parse_source_coordinates(SOURCE_COORDINATES)
SOURCE_SESSION_ID = SOURCE_CONFIG["sessionId"]
SOURCE_CAMERAS = SOURCE_CONFIG["cameras"]

print("Source sessionId:", SOURCE_SESSION_ID)
print("Source cameras:")
for camera in SOURCE_CAMERAS:
    print(f"- {camera['name']}: {camera['trackName']}")


## 4. Configure and load the trained RF-DETR Medium checkpoint

1. Put the trained checkpoint in Google Drive. The recommended inference file is normally `checkpoint_best_total.pth`.
2. Set `CHECKPOINT_PATH` to its exact Drive path.
3. Keep `TRUST_CHECKPOINT=True` only for a checkpoint you created or fully trust.
4. Leave `INFERENCE_SHAPE=None` to use the resolution stored by the model. A manual shape must satisfy RF-DETR's architecture divisibility constraints.
5. Leave `TARGET_CLASS_NAMES` empty to retain every checkpoint class.

The loader first uses `RFDETR.from_checkpoint()` so the architecture and class count are inferred from the checkpoint. If an older checkpoint lacks enough metadata, it falls back to `RFDETRMedium(pretrain_weights=...)`.


In [ ]:
# Mount Google Drive when the checkpoint is stored there.
USE_GOOGLE_DRIVE = True
DRIVE_MOUNT_POINT = "/content/drive"

# Change this to your trained RF-DETR Medium checkpoint.
CHECKPOINT_PATH = (
    "/content/drive/MyDrive/rfdetr/checkpoint_best_total.pth"
)

# Only enable for a checkpoint you created or fully trust.
TRUST_CHECKPOINT = True

CONFIDENCE = 0.35
MAX_DETECTIONS = 100
TARGET_CLASS_NAMES: list[str] = []  # Example: ["hot-wheels"]

# None uses the resolution reconstructed from the checkpoint.
# Example override for RF-DETR Medium: (576, 576)
INFERENCE_SHAPE: Optional[tuple[int, int]] = None

# RF-DETR's optional inference optimization. compile=False is more compatible
# with Colab and WebRTC's dynamic runtime. FP16 is suitable for CUDA inference.
OPTIMIZE_FOR_INFERENCE = True
OPTIMIZE_COMPILE = False
OPTIMIZE_INPLACE = True
USE_HALF_PRECISION = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

checkpoint_file = Path(CHECKPOINT_PATH).expanduser()
if not checkpoint_file.is_file():
    raise FileNotFoundError(
        f"RF-DETR checkpoint was not found: {checkpoint_file}"
    )

print(f"Loading RF-DETR checkpoint: {checkpoint_file}")

try:
    rf_model = RFDETR.from_checkpoint(
        checkpoint_file,
        trust_checkpoint=TRUST_CHECKPOINT,
    )
except ValueError as auto_detect_error:
    warnings.warn(
        "The checkpoint architecture could not be inferred automatically. "
        "Falling back to RFDETRMedium(pretrain_weights=...). "
        f"Original error: {auto_detect_error}"
    )
    rf_model = RFDETRMedium(
        pretrain_weights=str(checkpoint_file),
        trust_checkpoint=TRUST_CHECKPOINT,
    )

loaded_model_class = type(rf_model).__name__
if "Medium" not in loaded_model_class:
    warnings.warn(
        f"The checkpoint reconstructed {loaded_model_class}, not RFDETRMedium. "
        "Verify that CHECKPOINT_PATH points to the intended Medium checkpoint."
    )

MODEL_CLASS_NAMES = [str(name) for name in rf_model.class_names]
if not MODEL_CLASS_NAMES:
    raise RuntimeError("The checkpoint did not expose any class names.")

available_names = {name.casefold(): name for name in MODEL_CLASS_NAMES}
missing_target_names = [
    name for name in TARGET_CLASS_NAMES
    if name.casefold() not in available_names
]
if missing_target_names:
    raise ValueError(
        f"Unknown TARGET_CLASS_NAMES: {missing_target_names}. "
        f"Available classes: {MODEL_CLASS_NAMES}"
    )

NORMALIZED_TARGET_CLASS_NAMES: Optional[set[str]] = (
    {name.casefold() for name in TARGET_CLASS_NAMES}
    if TARGET_CLASS_NAMES
    else None
)

if OPTIMIZE_FOR_INFERENCE:
    rf_model.inference(
        compile=OPTIMIZE_COMPILE,
        batch_size=1,
        dtype=torch.float16 if USE_HALF_PRECISION else torch.float32,
        inplace=OPTIMIZE_INPLACE,
    )

# Warm up once so checkpoint initialization does not happen during signaling.
warmup_height, warmup_width = INFERENCE_SHAPE or (576, 576)
warmup_image = np.zeros(
    (warmup_height, warmup_width, 3),
    dtype=np.uint8,
)
warmup_kwargs: dict[str, Any] = {
    "threshold": CONFIDENCE,
    "include_source_image": False,
}
if INFERENCE_SHAPE is not None:
    warmup_kwargs["shape"] = INFERENCE_SHAPE

_ = rf_model.predict(warmup_image, **warmup_kwargs)

print("RF-DETR checkpoint loaded.")
print("Reconstructed class:", loaded_model_class)
print("Checkpoint:", checkpoint_file.name)
print("Classes:", MODEL_CLASS_NAMES)
print("Class filter:", TARGET_CLASS_NAMES or "all checkpoint classes")


## 5. Cloudflare Realtime API and WebRTC helpers

In [ ]:
class CloudflareRealtimeAPI:
    def __init__(
        self,
        app_id: str,
        app_secret: str,
        base_url: str = "https://rtc.live.cloudflare.com/v1",
    ):
        self.app_id = app_id
        self.app_secret = app_secret
        self.prefix = f"{base_url}/apps/{app_id}"
        self.session_id: Optional[str] = None

    async def _request(
        self,
        path: str,
        body: dict[str, Any],
        method: str = "POST",
    ) -> dict[str, Any]:
        url = f"{self.prefix}{path}"
        headers = {
            "content-type": "application/json",
            "authorization": f"Bearer {self.app_secret}",
        }
        timeout = aiohttp.ClientTimeout(total=30)
        async with aiohttp.ClientSession(timeout=timeout) as session:
            async with session.request(
                method,
                url,
                headers=headers,
                json=body,
            ) as response:
                text = await response.text()
                try:
                    result = json.loads(text)
                except Exception as exc:
                    raise RuntimeError(
                        f"Cloudflare returned HTTP {response.status}: {text[:500]}"
                    ) from exc

                if response.status < 200 or response.status >= 300:
                    raise RuntimeError(
                        f"Cloudflare returned HTTP {response.status}: {result}"
                    )
                self._check_errors(result)
                return result

    @staticmethod
    def _check_errors(result: dict[str, Any]) -> None:
        if result.get("errorCode"):
            raise RuntimeError(
                f"{result.get('errorCode')}: {result.get('errorDescription')}"
            )
        for index, track in enumerate(result.get("tracks", [])):
            if track.get("errorCode"):
                raise RuntimeError(
                    f"tracks[{index}] {track.get('errorCode')}: "
                    f"{track.get('errorDescription')}"
                )

    async def new_session(self, offer_sdp: str) -> dict[str, Any]:
        result = await self._request(
            "/sessions/new",
            {
                "sessionDescription": {
                    "type": "offer",
                    "sdp": offer_sdp,
                }
            },
        )
        self.session_id = result["sessionId"]
        return result

    async def new_tracks(
        self,
        tracks: list[dict[str, Any]],
        offer_sdp: Optional[str] = None,
    ) -> dict[str, Any]:
        if not self.session_id:
            raise RuntimeError("Cloudflare session has not been created.")
        body: dict[str, Any] = {"tracks": tracks}
        if offer_sdp is not None:
            body["sessionDescription"] = {
                "type": "offer",
                "sdp": offer_sdp,
            }
        return await self._request(
            f"/sessions/{self.session_id}/tracks/new",
            body,
        )

    async def renegotiate(self, answer_sdp: str) -> dict[str, Any]:
        if not self.session_id:
            raise RuntimeError("Cloudflare session has not been created.")
        return await self._request(
            f"/sessions/{self.session_id}/renegotiate",
            {
                "sessionDescription": {
                    "type": "answer",
                    "sdp": answer_sdp,
                }
            },
            method="PUT",
        )


def rtc_configuration() -> RTCConfiguration:
    return RTCConfiguration(
        iceServers=[RTCIceServer(urls=["stun:stun.cloudflare.com:3478"])]
    )


async def wait_for_ice_gathering(
    pc: RTCPeerConnection,
    timeout_seconds: float = 15.0,
) -> None:
    if pc.iceGatheringState == "complete":
        return

    event = asyncio.Event()

    @pc.on("icegatheringstatechange")
    async def _on_state_change() -> None:
        if pc.iceGatheringState == "complete":
            event.set()

    await asyncio.wait_for(event.wait(), timeout=timeout_seconds)


async def set_complete_local_description(
    pc: RTCPeerConnection,
    description: RTCSessionDescription,
) -> RTCSessionDescription:
    await pc.setLocalDescription(description)
    await wait_for_ice_gathering(pc)
    if pc.localDescription is None:
        raise RuntimeError("PeerConnection has no local description.")
    return pc.localDescription


async def wait_for_connection(
    pc: RTCPeerConnection,
    timeout_seconds: float = 20.0,
) -> None:
    if pc.iceConnectionState in {"connected", "completed"}:
        return

    event = asyncio.Event()

    @pc.on("iceconnectionstatechange")
    async def _on_ice_state() -> None:
        state = pc.iceConnectionState
        print("ICE state:", state)
        if state in {"connected", "completed", "failed", "closed"}:
            event.set()

    await asyncio.wait_for(event.wait(), timeout=timeout_seconds)
    if pc.iceConnectionState not in {"connected", "completed"}:
        raise RuntimeError(f"WebRTC connection failed: {pc.iceConnectionState}")


def description_from_result(result: dict[str, Any]) -> RTCSessionDescription:
    description = result["sessionDescription"]
    return RTCSessionDescription(
        sdp=description["sdp"],
        type=description["type"],
    )


## 6. Low-latency multi-camera RF-DETR Medium tracks

In [ ]:
@dataclass
class CameraPipelineStats:
    camera_name: str
    source_track_name: str
    output_track_name: str
    received_frames: int = 0
    processed_frames: int = 0
    last_inference_ms: float = 0.0
    last_object_count: int = 0
    started_at: float = 0.0

    @property
    def uptime_seconds(self) -> float:
        return max(0.0, time.monotonic() - self.started_at)

    @property
    def processed_fps(self) -> float:
        if self.uptime_seconds <= 0:
            return 0.0
        return self.processed_frames / self.uptime_seconds


class LatestFrameBuffer:
    """Continuously consumes a source and retains only its newest frame."""

    def __init__(self, source: MediaStreamTrack, stats: CameraPipelineStats):
        self.source = source
        self.stats = stats
        self._latest: Optional[av.VideoFrame] = None
        self._sequence = 0
        self._event = asyncio.Event()
        self._exception: Optional[BaseException] = None
        self._task = asyncio.create_task(self._reader())

    async def _reader(self) -> None:
        try:
            while True:
                frame = await self.source.recv()
                self.stats.received_frames += 1
                self._latest = frame
                self._sequence += 1
                self._event.set()
        except BaseException as exc:
            self._exception = exc
            self._event.set()

    async def newest_after(
        self,
        previous_sequence: int,
    ) -> tuple[int, av.VideoFrame]:
        while True:
            if self._exception is not None:
                raise self._exception
            if self._latest is not None and self._sequence > previous_sequence:
                return self._sequence, self._latest
            self._event.clear()
            await self._event.wait()

    async def close(self) -> None:
        self._task.cancel()
        try:
            await self._task
        except BaseException:
            pass


class RFDETRStreamingEngine:
    """A single shared RF-DETR model used safely by all camera tracks."""

    def __init__(
        self,
        model: RFDETR,
        confidence: float,
        max_detections: int,
        target_class_names: Optional[set[str]],
        inference_shape: Optional[tuple[int, int]],
    ):
        self.model = model
        self.confidence = confidence
        self.max_detections = max_detections
        self.target_class_names = target_class_names
        self.inference_shape = inference_shape
        self._lock = threading.Lock()

    def process(
        self,
        rgb: np.ndarray,
        camera_name: str,
    ) -> tuple[np.ndarray, float, int]:
        # RF-DETR expects NumPy images in RGB channel order. It returns boxes in
        # the source image coordinate system, so annotation can stay full-size.
        predict_kwargs: dict[str, Any] = {
            "threshold": self.confidence,
            "include_source_image": False,
        }
        if self.inference_shape is not None:
            predict_kwargs["shape"] = self.inference_shape

        # A shared model is serialized because simultaneous predict() calls on
        # one instance are not assumed to be thread-safe.
        with self._lock:
            started_at = time.perf_counter()
            detections = self.model.predict(
                np.ascontiguousarray(rgb),
                **predict_kwargs,
            )
            latency_ms = (time.perf_counter() - started_at) * 1000.0

        boxes = np.asarray(detections.xyxy, dtype=np.float32)
        detection_count = len(boxes)

        if detections.confidence is None:
            confidences = np.ones(detection_count, dtype=np.float32)
        else:
            confidences = np.asarray(
                detections.confidence,
                dtype=np.float32,
            )

        if detections.class_id is None:
            class_ids = np.full(detection_count, -1, dtype=np.int32)
        else:
            class_ids = np.asarray(detections.class_id, dtype=np.int32)

        raw_class_names = detections.data.get("class_name")
        if raw_class_names is not None and len(raw_class_names) == detection_count:
            class_names = np.asarray(
                [str(name) for name in raw_class_names],
                dtype=object,
            )
        else:
            class_names = np.asarray(
                [
                    MODEL_CLASS_NAMES[class_id]
                    if 0 <= class_id < len(MODEL_CLASS_NAMES)
                    else f"class-{class_id}"
                    for class_id in class_ids
                ],
                dtype=object,
            )

        if self.target_class_names is not None and detection_count:
            class_mask = np.asarray(
                [
                    str(name).casefold() in self.target_class_names
                    for name in class_names
                ],
                dtype=bool,
            )
            boxes = boxes[class_mask]
            confidences = confidences[class_mask]
            class_ids = class_ids[class_mask]
            class_names = class_names[class_mask]

        if len(boxes) > self.max_detections:
            selected = np.argsort(confidences)[::-1][: self.max_detections]
            boxes = boxes[selected]
            confidences = confidences[selected]
            class_ids = class_ids[selected]
            class_names = class_names[selected]

        object_count = len(boxes)
        image = Image.fromarray(rgb.copy())
        draw = ImageDraw.Draw(image)
        font = ImageFont.load_default()
        image_width, image_height = image.size

        for box, confidence, class_name in zip(
            boxes,
            confidences,
            class_names,
        ):
            x1, y1, x2, y2 = [float(value) for value in box]
            x1 = int(max(0, min(image_width - 1, round(x1))))
            y1 = int(max(0, min(image_height - 1, round(y1))))
            x2 = int(max(0, min(image_width - 1, round(x2))))
            y2 = int(max(0, min(image_height - 1, round(y2))))

            label = f"{class_name} {float(confidence):.2f}"
            label_y = max(0, y1 - 16)
            text_box = draw.textbbox((x1, label_y), label, font=font)

            draw.rectangle((x1, y1, x2, y2), outline=(255, 60, 60), width=3)
            draw.rectangle(
                (
                    text_box[0] - 2,
                    text_box[1] - 2,
                    text_box[2] + 2,
                    text_box[3] + 2,
                ),
                fill=(255, 60, 60),
            )
            draw.text(
                (x1, label_y),
                label,
                fill=(255, 255, 255),
                font=font,
            )

        status = (
            f"RF-DETR M | {camera_name} | objects: {object_count} | "
            f"inference: {latency_ms:.1f} ms"
        )
        status_box = draw.textbbox((8, 8), status, font=font)
        draw.rectangle(
            (4, 4, status_box[2] + 12, status_box[3] + 12),
            fill=(0, 0, 0),
        )
        draw.text((8, 8), status, fill=(255, 255, 255), font=font)

        return np.asarray(image), latency_ms, object_count


class RFDETRProcessedVideoTrack(MediaStreamTrack):
    kind = "video"

    def __init__(
        self,
        source_buffer: LatestFrameBuffer,
        engine: RFDETRStreamingEngine,
        stats: CameraPipelineStats,
    ):
        super().__init__()
        self.source_buffer = source_buffer
        self.engine = engine
        self.stats = stats
        self._last_sequence = 0

    async def recv(self) -> av.VideoFrame:
        sequence, source_frame = await self.source_buffer.newest_after(
            self._last_sequence
        )
        self._last_sequence = sequence

        source_rgb = source_frame.to_ndarray(format="rgb24")
        annotated_rgb, latency_ms, object_count = await asyncio.to_thread(
            self.engine.process,
            source_rgb,
            self.stats.camera_name,
        )

        self.stats.processed_frames += 1
        self.stats.last_inference_ms = latency_ms
        self.stats.last_object_count = object_count

        output_frame = av.VideoFrame.from_ndarray(annotated_rgb, format="rgb24")
        output_frame.pts = source_frame.pts
        output_frame.time_base = source_frame.time_base
        return output_frame


## 7. Subscribe to every source track and publish all RF-DETR outputs

In [ ]:
async def subscribe_to_cloudflare_video(
    app_id: str,
    app_secret: str,
    source_session_id: str,
    source_track_name: str,
) -> tuple[RTCPeerConnection, CloudflareRealtimeAPI, MediaStreamTrack]:
    api = CloudflareRealtimeAPI(app_id, app_secret)
    pc = RTCPeerConnection(rtc_configuration())
    loop = asyncio.get_running_loop()
    video_future: asyncio.Future[MediaStreamTrack] = loop.create_future()

    @pc.on("track")
    def _on_track(track: MediaStreamTrack) -> None:
        print(
            f"Received source track {source_track_name}: "
            f"kind={track.kind}, id={track.id}"
        )
        if track.kind == "video" and not video_future.done():
            video_future.set_result(track)

    # Bootstrap media is used only to establish the Cloudflare WebRTC session.
    bootstrap_track = VideoStreamTrack()
    pc.addTransceiver(bootstrap_track, direction="sendonly")

    offer = await pc.createOffer()
    local_offer = await set_complete_local_description(pc, offer)
    session_result = await api.new_session(local_offer.sdp)
    await pc.setRemoteDescription(description_from_result(session_result))
    await wait_for_connection(pc)

    pull_result = await api.new_tracks(
        [
            {
                "location": "remote",
                "sessionId": source_session_id,
                "trackName": source_track_name,
            }
        ]
    )

    if pull_result.get("requiresImmediateRenegotiation"):
        description = pull_result.get("sessionDescription")
        if not description or description.get("type") != "offer":
            raise RuntimeError(
                "Cloudflare requested renegotiation but did not return an offer."
            )
        await pc.setRemoteDescription(
            RTCSessionDescription(
                sdp=description["sdp"],
                type=description["type"],
            )
        )
        answer = await pc.createAnswer()
        local_answer = await set_complete_local_description(pc, answer)
        await api.renegotiate(local_answer.sdp)
    elif pull_result.get("sessionDescription"):
        await pc.setRemoteDescription(description_from_result(pull_result))

    remote_video = await asyncio.wait_for(video_future, timeout=20.0)
    return pc, api, remote_video


def slugify_track_part(value: str) -> str:
    slug = re.sub(r"[^a-zA-Z0-9_-]+", "-", value.strip()).strip("-").lower()
    return slug or "camera"


async def publish_cloudflare_videos(
    app_id: str,
    app_secret: str,
    processed_tracks: list[dict[str, Any]],
) -> tuple[RTCPeerConnection, CloudflareRealtimeAPI, str]:
    if not processed_tracks:
        raise ValueError("At least one processed track is required.")

    api = CloudflareRealtimeAPI(app_id, app_secret)
    pc = RTCPeerConnection(rtc_configuration())

    transceivers = []
    for item in processed_tracks:
        transceiver = pc.addTransceiver(item["track"], direction="sendonly")
        transceivers.append((item, transceiver))

    offer = await pc.createOffer()
    local_offer = await set_complete_local_description(pc, offer)
    session_result = await api.new_session(local_offer.sdp)
    await pc.setRemoteDescription(description_from_result(session_result))
    await wait_for_connection(pc)

    local_track_objects = []
    for item, transceiver in transceivers:
        if transceiver.mid is None:
            raise RuntimeError(
                f"Cloudflare did not assign a media MID for {item['track_name']}."
            )
        local_track_objects.append(
            {
                "location": "local",
                "mid": transceiver.mid,
                "trackName": item["track_name"],
            }
        )

    registration_offer = await pc.createOffer()
    local_registration_offer = await set_complete_local_description(
        pc,
        registration_offer,
    )
    tracks_result = await api.new_tracks(
        local_track_objects,
        offer_sdp=local_registration_offer.sdp,
    )
    await pc.setRemoteDescription(description_from_result(tracks_result))

    if not api.session_id:
        raise RuntimeError("Processed publisher session ID is missing.")
    return pc, api, api.session_id


async def start_multi_camera_pipeline() -> dict[str, Any]:
    engine = RFDETRStreamingEngine(
        model=rf_model,
        confidence=CONFIDENCE,
        max_detections=MAX_DETECTIONS,
        target_class_names=NORMALIZED_TARGET_CLASS_NAMES,
        inference_shape=INFERENCE_SHAPE,
    )

    run_suffix = uuid.uuid4().hex[:8]
    camera_pipelines: list[dict[str, Any]] = []
    publisher_pc: Optional[RTCPeerConnection] = None

    try:
        for index, camera in enumerate(SOURCE_CAMERAS, start=1):
            name = camera["name"]
            source_track_name = camera["trackName"]
            output_track_name = (
                f"rfdetr-m-{index}-{slugify_track_part(name)}-{run_suffix}"
            )

            print(
                f"[{index}/{len(SOURCE_CAMERAS)}] Subscribing to "
                f"{name} / {source_track_name}..."
            )
            subscriber_pc, subscriber_api, source_track = (
                await subscribe_to_cloudflare_video(
                    APP_ID,
                    APP_SECRET,
                    SOURCE_SESSION_ID,
                    source_track_name,
                )
            )

            stats = CameraPipelineStats(
                camera_name=name,
                source_track_name=source_track_name,
                output_track_name=output_track_name,
                started_at=time.monotonic(),
            )
            frame_buffer = LatestFrameBuffer(source_track, stats)
            processed_track = RFDETRProcessedVideoTrack(
                source_buffer=frame_buffer,
                engine=engine,
                stats=stats,
            )

            camera_pipelines.append(
                {
                    "camera": camera,
                    "subscriber_pc": subscriber_pc,
                    "subscriber_api": subscriber_api,
                    "source_track": source_track,
                    "frame_buffer": frame_buffer,
                    "processed_track": processed_track,
                    "stats": stats,
                    "output_track_name": output_track_name,
                }
            )
            print(f"Connected: {name}")

        print(f"Publishing {len(camera_pipelines)} processed RF-DETR Medium tracks...")
        publish_items = [
            {
                "track": item["processed_track"],
                "track_name": item["output_track_name"],
            }
            for item in camera_pipelines
        ]
        publisher_pc, publisher_api, result_session_id = (
            await publish_cloudflare_videos(
                APP_ID,
                APP_SECRET,
                publish_items,
            )
        )

        pipeline_id = f"rfdetr-m-{run_suffix}"
        result_coordinates = {
            # Backwards-compatible field used by the current HTML viewers.
            "sessionId": result_session_id,
            # Stable application-level fields suitable for a Next.js backend.
            "pipelineId": pipeline_id,
            "status": "running",
            "model": {
                "architecture": loaded_model_class,
                "checkpoint": checkpoint_file.name,
                "confidence": CONFIDENCE,
                "targetClassNames": TARGET_CLASS_NAMES,
            },
            "sourceSessionId": SOURCE_SESSION_ID,
            "processedSessionId": result_session_id,
            "results": [
                {
                    "cameraName": item["camera"]["name"],
                    # Backwards-compatible flattened fields.
                    "sourceTrackName": item["camera"]["trackName"],
                    "processedSessionId": result_session_id,
                    "processedTrackName": item["output_track_name"],
                    # Preferred nested representation for Next.js.
                    "source": {
                        "sessionId": SOURCE_SESSION_ID,
                        "trackName": item["camera"]["trackName"],
                    },
                    "processed": {
                        "sessionId": result_session_id,
                        "trackName": item["output_track_name"],
                    },
                }
                for item in camera_pipelines
            ],
        }

        pipeline = {
            "engine": engine,
            "camera_pipelines": camera_pipelines,
            "publisher_pc": publisher_pc,
            "publisher_api": publisher_api,
            "result_session_id": result_session_id,
            "result_coordinates": result_coordinates,
        }

        print("\n" + "=" * 88)
        print("PASTE THESE VALUES INTO THE HTML PROCESSED RESULT VIEWERS")
        print("=" * 88)
        print("Processed sessionId (same for every viewer):", result_session_id)
        for index, item in enumerate(result_coordinates["results"], start=1):
            print(
                f"Viewer {index} | {item['cameraName']} | "
                f"trackName: {item['processedTrackName']}"
            )
        print("=" * 88)
        print(json.dumps(result_coordinates, indent=2))
        return pipeline

    except BaseException:
        for item in camera_pipelines:
            await item["frame_buffer"].close()
            await item["subscriber_pc"].close()
        if publisher_pc is not None:
            await publisher_pc.close()
        raise


async def stop_multi_camera_pipeline(
    pipeline: Optional[dict[str, Any]],
) -> None:
    if not pipeline:
        return

    for item in pipeline.get("camera_pipelines", []):
        frame_buffer = item.get("frame_buffer")
        if frame_buffer is not None:
            await frame_buffer.close()
        subscriber_pc = item.get("subscriber_pc")
        if subscriber_pc is not None:
            await subscriber_pc.close()

    publisher_pc = pipeline.get("publisher_pc")
    if publisher_pc is not None:
        await publisher_pc.close()

    print("Multi-camera pipeline stopped.")


## 8. Start the multi-camera pipeline

Keep the HTML page open and publishing while this cell runs. If rerunning after a previous start, run the cleanup cell first.

The final JSON contains both:

- flattened `processedSessionId` / `processedTrackName` values for the current HTML;
- nested `source` / `processed` objects and pipeline metadata for a Next.js backend.


In [ ]:
try:
    if PIPELINE is not None:
        await stop_multi_camera_pipeline(PIPELINE)
except NameError:
    pass

PIPELINE = await start_multi_camera_pipeline()


## 9. Inspect all camera pipelines

In [ ]:
if not PIPELINE:
    raise RuntimeError("Start the pipeline first.")

print("Processed sessionId:", PIPELINE["result_session_id"])
print("Publisher ICE state:", PIPELINE["publisher_pc"].iceConnectionState)
print()

for index, item in enumerate(PIPELINE["camera_pipelines"], start=1):
    stats = item["stats"]
    print(
        {
            "index": index,
            "camera": stats.camera_name,
            "source_track": stats.source_track_name,
            "processed_track": stats.output_track_name,
            "received_frames": stats.received_frames,
            "processed_frames": stats.processed_frames,
            "processed_fps_average": round(stats.processed_fps, 3),
            "last_inference_ms": round(stats.last_inference_ms, 1),
            "last_object_count": stats.last_object_count,
            "subscriber_ice_state": item["subscriber_pc"].iceConnectionState,
        }
    )


## 10. Stop and clean up

Run this before changing the source coordinates, model, or camera set.


In [ ]:
await stop_multi_camera_pipeline(PIPELINE)
PIPELINE = None


## Troubleshooting

**Cloudflare returns unauthorized**

- Rotate any previously exposed secret.
- Confirm the App ID and App Secret belong to the same Cloudflare Realtime application.
- Keep the secret out of screenshots and shared notebooks.

**Checkpoint not found**

- Confirm Google Drive is mounted.
- Use the full path beginning with `/content/drive/MyDrive/`.
- Prefer the trained inference checkpoint, usually `checkpoint_best_total.pth`.

**Checkpoint cannot be inferred**

- The notebook falls back to `RFDETRMedium(pretrain_weights=...)` for older Medium checkpoints.
- Confirm the checkpoint really came from RF-DETR Medium training.
- Set `TRUST_CHECKPOINT=True` only when the file is trusted.

**Incorrect or missing class names**

- New RF-DETR checkpoints normally embed class names.
- Check `MODEL_CLASS_NAMES` after the loading cell.
- Use exact class spellings in `TARGET_CLASS_NAMES`.

**Track not found**

- Keep the HTML page open and publishing.
- Copy a fresh JSON/ENV block after clicking **Start and publish all cameras**.
- Session IDs and track names are case-sensitive.

**The HTML only shows one processed stream**

- Add one result viewer for every camera.
- Use the same processed `sessionId` in every viewer.
- Use the corresponding unique processed `trackName` printed by the start cell.

**CUDA out of memory**

- Reduce the number of active cameras.
- Set `OPTIMIZE_INPLACE=True` so the optimized inference model replaces the original model in memory.
- Disable inference compilation with `OPTIMIZE_COMPILE=False`.
- Use the checkpoint's default resolution by keeping `INFERENCE_SHAPE=None`.
- Stop old pipelines before starting a new one.

**A manual inference shape fails**

- Keep `INFERENCE_SHAPE=None`, or use a shape compatible with the trained RF-DETR architecture.
- RF-DETR Medium commonly uses `576 × 576`, but the checkpoint's stored configuration is safer.

**Latency grows with more cameras**

- The notebook intentionally shares one model and serializes inference for safety.
- Each camera drops stale frames, preventing a growing queued backlog, but per-camera output FPS decreases as camera count increases.
- Increase `CONFIDENCE`, reduce `MAX_DETECTIONS`, filter `TARGET_CLASS_NAMES`, or reduce the number of cameras.
- For production, use a persistent GPU service and consider one inference worker per GPU.

**No audio**

- The notebook republishes annotated video only. The browser can keep using its original audio path separately when needed.
